<a href="https://colab.research.google.com/github/AmaadZiaGit/Machine-Learning-/blob/main/NVD_Dataset_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. LOAD & CLEAN
df = pd.read_csv('NVD_Cybersecurity_Dataset.csv', on_bad_lines='skip', engine='python')
df = df.dropna(subset=['Description', 'CVSS_Base_Score'])

# 2. PREPROCESS
# We use a smaller sample (e.g., 20,000 rows) so it runs FAST for your redo
df_sample = df.sample(20000, random_state=42)
tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
X = tfidf.fit_transform(df_sample['Description'])
y = df_sample['CVSS_Base_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. THE 4 TECHNIQUES
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=10),
    "K-Neighbors": KNeighborsRegressor(n_neighbors=5)
}

# 4. TRAIN AND EVALUATE
print(f"{'Model':<20} | {'MAE':<10} | {'R2 Score':<10}")
print("-" * 45)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name:<20} | {mae:<10.3f} | {r2:<10.3f}")

    # Save the best one (usually Random Forest)
    if name == "Random Forest":
        joblib.dump(model, 'cve_regression_model.pkl')

joblib.dump(tfidf, 'tfidf_reg_vectorizer.pkl')

Model                | MAE        | R2 Score  
---------------------------------------------
Linear Regression    | 1.054      | 0.542     
Decision Tree        | 1.005      | 0.541     
Random Forest        | 0.990      | 0.569     
K-Neighbors          | 1.092      | 0.424     


['tfidf_reg_vectorizer.pkl']